In [0]:
%pip install statsmodels

"""
Feature_Engineering_10_Window_Dataset.py

Python equivalent of Feature_Engineering_10_Window_Dataset.R

Loads the windowed EDA data, computes row-wise summary features (max, min,
std, kurtosis, skewness, mean) over the raw time-series columns, saves the
result, produces boxplots of each feature by condition, and fits a
multinomial logistic regression of condition on the computed features.

Requires: pandas, numpy, scipy, matplotlib, seaborn, statsmodels
    pip install pandas numpy scipy matplotlib seaborn statsmodels
"""

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import kurtosis as sp_kurtosis
from scipy.stats import norm
from scipy.stats import skew as sp_skew
from statsmodels.formula.api import mnlogit

# ---------------------------------------------------------------------------
# Set working directory to the folder this script lives in
# (equivalent of the rstudioapi current_path / setwd() calls in the R script)
# ---------------------------------------------------------------------------
#current_path = Path(__file__).resolve().parent
#os.chdir(current_path)

df = spark.table("eda.default.eda_silver_windowed").toPandas()

window_size = 10
fq = 4

# ---------------------------------------------------------------------------
# R: df[2:(window_size*fq - 1)] selects columns 2..39 (1-based, i.e. R's
# 38 raw value columns), skipping the row-label column (col 1) and NOT
# including the last two raw value columns / condition / source_rows.
# In pandas 0-based iloc this is columns[1:39] (38 columns).
# ---------------------------------------------------------------------------
raw_cols = df.iloc[:, 1:(window_size * fq - 1)]


def row_kurtosis(row):
    vals = row.dropna().to_numpy()
    if len(vals) < 2:
        return np.nan
    # moments::kurtosis() returns the regular (non-excess) kurtosis using
    # biased (population) moments -> fisher=False, bias=True in scipy
    return sp_kurtosis(vals, fisher=False, bias=True)


def row_skewness(row):
    vals = row.dropna().to_numpy()
    if len(vals) < 2:
        return np.nan
    # moments::skewness() uses biased (population) moments -> bias=True in scipy
    return sp_skew(vals, bias=True)


df["Max_raw"] = raw_cols.max(axis=1, skipna=True)
df["Min_raw"] = raw_cols.min(axis=1, skipna=True)
df["std_raw"] = raw_cols.std(axis=1, skipna=True, ddof=1)  # R's sd() uses n-1
df["kurtosis_raw"] = raw_cols.apply(row_kurtosis, axis=1)
df["skewness_raw"] = raw_cols.apply(row_skewness, axis=1)
df["mean_raw"] = raw_cols.mean(axis=1, skipna=True)

df.to_csv("../../Data/Processed/Raw_Features.csv", index=True)

spark_features = spark.createDataFrame(df)
spark_features.write.format("delta").mode("overwrite").saveAsTable("eda.default.eda_gold_features")

# ---------------------------------------------------------------------------
# Boxplots of each feature by condition (equivalent of the ggplot2 blocks)
# ---------------------------------------------------------------------------
condition_levels = ["neutral", "positive", "negative", "black"]
df["condition"] = pd.Categorical(df["condition"], categories=condition_levels)

feature_titles = {
    "Max_raw": "Spread of Max values for Each Condition",
    "Min_raw": "Spread of Min values for Each Condition",
    "std_raw": "Spread of Standard Deviation values for Each Condition",
    "kurtosis_raw": "Spread of Kurtosis values for Each Condition",
    "skewness_raw": "Spread of Skewness values for Each Condition",
    "mean_raw": "Spread of mean values for Each Condition",
}

for feature, title in feature_titles.items():
    plt.figure(figsize=(7, 5))
    sns.boxplot(data=df, x="condition", y=feature, hue="condition",
                order=condition_levels, legend=False)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f"../../Data/Processed/boxplot_{feature}.png", dpi=150)
    plt.close()

# ---------------------------------------------------------------------------
# Multinomial logistic regression (equivalent of nnet::multinom)
# "neutral" is first in condition_levels, so it's used as the reference
# category, matching R's factor level ordering behavior.
# -------------------------------------------------------------